# Add the new patients(already cropped- for the cropped go to: /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/comprobate_labels.ipynb in global cropped aligned) to the csv for the densenet

In [2]:
import pandas as pd
import os

# --- CONFIGURATION ---
SOURCE_CSV = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/filtered_midas900_t2w_partition.csv"
TARGET_CSV = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/new_input_data.csv"
CROPPED_ROOT = "/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/old_patients/"

# CORRECTED MAPPING based on your specific column order:
# Column 1 = L5-S1, Column 5 = L1-L2
DISC_LOOKUP = {
    "1": "L5-S1",
    "2": "L4-L5",
    "3": "L3-L4",
    "4": "L2-L3",
    "5": "L1-L2"
}

def create_new_entries():
    df_src = pd.read_csv(SOURCE_CSV)
    new_rows = []

    print(f"Processing {len(df_src)} subjects from source...")

    for _, row in df_src.iterrows():
        subj_mids = str(row['Subject_MIDS'])
        ses_mids = str(row['Session_MIDS'])
        t2_full_path = row['Image']
        xnat_name = row['Session_XNAT']
        
        for col_id, disc_label in DISC_LOOKUP.items():
            # Get Pfirrmann value from columns '1' through '5'
            pfirrmann_val = row[col_id]
            
            # Construct path to the cropped file
            # Pattern matches previous script: {CROPPED_ROOT}/{Subject}/{Subject}_{Disc}.nii.gz
            cropped_filename = f"{subj_mids}_{disc_label}.nii.gz"
            cropped_path = os.path.join(CROPPED_ROOT, subj_mids, cropped_filename)
            
            if os.path.exists(cropped_path):
                new_entry = {
                    'subj': subj_mids.replace('sub-', ''),
                    'sess': ses_mids.replace('ses-', ''),
                    'image_T2': t2_full_path,
                    'image_T1': None, 
                    'disc': disc_label,
                    'disc_path': cropped_path,
                    'XNAT_name': xnat_name,
                    'Pfirrmann': pfirrmann_val
                }
                new_rows.append(new_entry)

    df_new = pd.DataFrame(new_rows)

    if os.path.exists(TARGET_CSV):
        df_target = pd.read_csv(TARGET_CSV)
        # Combine existing and new, then drop duplicates if necessary
        df_final = pd.concat([df_target, df_new], ignore_index=True)
        print(f"Appending {len(df_new)} rows to {TARGET_CSV}")
    else:
        df_final = df_new
        print(f"Creating new file at {TARGET_CSV}")

    df_final.to_csv(TARGET_CSV, index=False)
    print("Process complete.")

if __name__ == "__main__":
    create_new_entries()

Processing 717 subjects from source...
Appending 3580 rows to /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/new_input_data.csv
Process complete.
